https://docs.pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial

In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [ ]:
from pathlib import Path
import pandas as pd
import polars as pl
from tabseq_torch.dl.encoder import Encoder, Cat2Code, EncodedDataCatalog

# Init data catalog

In [ ]:
data_path = "~/data/tbts_data"
! mkdir -p $data_path
data_path = ! realpath $data_path
data_path = Path(data_path[0])

In [ ]:
data_path

In [ ]:
outputfile_name = "tbts-names"

In [ ]:
path_to_raw_data = "~/data/tbts_data/raw/names/"
! mkdir -p $path_to_raw_data
path_to_raw_data = ! find $path_to_raw_data
path_to_raw_data = path_to_raw_data[0]

In [ ]:
tbts_path = data_path / Path(f"{outputfile_name}.parquet")

In [ ]:
labels_col_name = "labels"
labels_path = data_path / Path(f"{labels_col_name}.parquet")

# Load Data

In [ ]:

! wget --no-check-certificate -P $path_to_raw_data https://download.pytorch.org/tutorial/data.zip

In [ ]:
! ls $path_to_raw_data

! unzip $path_to_raw_data/'data.zip' -d $path_to_raw_data


# Create dataframes

In [ ]:
g_col_name = "G"
cat_col_name = "name_chars"
T = "T"
path_to_raw_data = Path(path_to_raw_data)
path = path_to_raw_data / "data" / "names"

In [ ]:
! ls $path

In [ ]:
def load_labeled_names(
    names_data_path: Path, file_names: list[str]
) -> tuple[list[list[str]], list[str]]:
    labels: list[str] = []
    name_cahrs: list[list[str]] = []
    for fnanme in file_names:
        with open(names_data_path / fnanme, mode="r", encoding="utf-8") as f:
            for line in f.read().strip().split("\n"):
                name_cahrs.append(list(line))
                labels.append(fnanme)
    return name_cahrs, labels

In [ ]:
file_names = sorted([f_path.name for f_path in path.glob("*.txt")])
df = pd.DataFrame(dict(zip([cat_col_name, labels_col_name],  load_labeled_names(names_data_path=path, file_names=file_names))))

In [ ]:
df.head()

In [ ]:
df[[labels_col_name]].head()

In [ ]:
labels_path = data_path / Path(f'{labels_col_name}.parquet')
labels_path

In [ ]:
df_labels =  df[[labels_col_name]]
df_labels[g_col_name] = df.index
df_labels.to_parquet(labels_path , engine='pyarrow')
df_labels.head()

In [ ]:
df_tbts =df.loc[:, df.columns != 'labels'].explode(cat_col_name).reset_index().rename(columns={'index': g_col_name})
df_tbts.head(10)

In [ ]:
df_tbts[T] = df_tbts.index
df_tbts[T] = df_tbts.groupby(g_col_name)[T].rank(method='dense')
df_tbts['F'] = df_tbts[[cat_col_name]].groupby(cat_col_name)[cat_col_name].transform('count')

In [ ]:
df_tbts

In [ ]:

ch_rank = df_tbts[[cat_col_name]].groupby(cat_col_name)[cat_col_name].agg('count').sort_values().rank(method='dense')

In [ ]:
import json

In [ ]:
! ls $data_path

In [ ]:
#with open(data_path / 'vowels.json', 'w') as f:
#    json.dump({c: '_' for c in ch_rank.index}, f, indent=4)

In [ ]:
with open(data_path / 'vowels.json', 'r') as f:
    d = json.load(f)
df_vc = pd.DataFrame({cat_col_name: d.keys(), "vc": d.values()})
df_vc.head()

In [ ]:
df_tbts_wiht_num_features =  pd.merge(df_tbts , ch_rank.to_frame().rename(columns={cat_col_name: 'F_Rank'}), left_on=cat_col_name, right_index=True)

In [ ]:
df_tbts_wiht_num_features = pd.merge(df_tbts_wiht_num_features, df_vc, left_on=cat_col_name, right_on=cat_col_name)

In [ ]:
df_tbts_wiht_num_features['index'] = df_tbts_wiht_num_features.index

In [ ]:
df_tbts_wiht_num_features

In [ ]:
df_tbts_wiht_num_features.to_parquet(path=tbts_path , engine='pyarrow')

# Tabts data validation

In [ ]:
output_files_path = data_path / Path(f"{outputfile_name}")
! ls -lah $output_files_path*

In [ ]:
pd.read_parquet(path=tbts_path, engine="pyarrow").groupby(g_col_name).ngroups

In [ ]:
pd.read_parquet(path=tbts_path, engine="pyarrow")

In [ ]:
pd.read_parquet(path=labels_path, engine="pyarrow")

In [ ]:
pl.scan_parquet(labels_path).head().collect()

In [ ]:
labels_path

# Encoding

In [ ]:
tbts_path.exists()